In [ ]:
import math
import urllib.request
import numpy as np
from collections import defaultdict, Counter


# ============================================================
# 1. DOWNLOAD DATASET
# ============================================================

TRAIN_URL = (
    "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/"
    "master/en_ewt-ud-train.conllu"
)

TEST_URL = (
    "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/"
    "master/en_ewt-ud-test.conllu"
)


def download_file(url, filename):
    try:
        urllib.request.urlretrieve(url, filename)
        print(f"Downloaded: {filename}")
    except Exception as e:
        print("Error downloading file:", e)


download_file(TRAIN_URL, "en_ewt-ud-train.conllu")
download_file(TEST_URL, "en_ewt-ud-test.conllu")


# ============================================================
# 2. LOAD CONLLU DATA
# ============================================================

def load_conllu(filename):

    sentences = []
    words = []
    tags = []

    with open(filename, "r", encoding="utf-8") as file:

        for line in file:

            line = line.strip()

            # End of sentence
            if not line:

                if words:
                    sentences.append((words, tags))
                    words = []
                    tags = []

                continue

            # Ignore comments
            if line.startswith("#"):
                continue

            columns = line.split("\t")

            # CoNLL-U must have 10 columns
            if len(columns) != 10:
                continue

            token_id = columns[0]

            # Ignore multi-word tokens and empty nodes
            if "-" in token_id or "." in token_id:
                continue

            word = columns[1]
            pos_tag = columns[3]

            words.append(word)
            tags.append(pos_tag)

        # Add final sentence
        if words:
            sentences.append((words, tags))

    return sentences


train_data = load_conllu("en_ewt-ud-train.conllu")
test_data = load_conllu("en_ewt-ud-test.conllu")


print("\nTraining sentences:", len(train_data))
print("Testing sentences:", len(test_data))


# ============================================================
# 3. BUILD HMM COUNTS
# ============================================================

transition_counts = defaultdict(Counter)
emission_counts = defaultdict(Counter)
tag_counts = Counter()

START = "<START>"
END = "<END>"


for words, tags in train_data:

    previous_tag = START

    for word, tag in zip(words, tags):

        # Transition count
        transition_counts[previous_tag][tag] += 1

        # Emission count
        emission_counts[tag][word.lower()] += 1

        # Tag count
        tag_counts[tag] += 1

        previous_tag = tag

    # Last tag -> END
    transition_counts[previous_tag][END] += 1


# ============================================================
# 4. POS TAG LIST
# ============================================================

tags_list = list(tag_counts.keys())

print("\nPOS Tags:")
print(tags_list)

num_tags = len(tags_list)


# ============================================================
# 5. CREATE TAG INDEX
# ============================================================

tag_to_index = {
    tag: i for i, tag in enumerate(tags_list)
}

index_to_tag = {
    i: tag for tag, i in tag_to_index.items()
}


# ============================================================
# 6. CREATE VOCABULARY
# ============================================================

vocabulary = set()

for words, tags in train_data:

    for word in words:
        vocabulary.add(word.lower())


vocabulary_size = len(vocabulary) + 1


print("\nVocabulary size:", vocabulary_size)


# ============================================================
# 7. BUILD TRANSITION PROBABILITY MATRIX
# ============================================================

transition_matrix = np.zeros((num_tags, num_tags))


for previous_tag in tags_list:

    previous_index = tag_to_index[previous_tag]

    total = sum(transition_counts[previous_tag].values())

    for current_tag in tags_list:

        current_index = tag_to_index[current_tag]

        # Laplace smoothing
        numerator = transition_counts[previous_tag][current_tag] + 1

        denominator = total + num_tags + 1

        transition_matrix[previous_index][current_index] = (
            numerator / denominator
        )


# ============================================================
# 8. START PROBABILITY
# ============================================================

start_probability = np.zeros(num_tags)


total_start = sum(transition_counts[START].values())


for tag in tags_list:

    index = tag_to_index[tag]

    numerator = transition_counts[START][tag] + 1

    denominator = total_start + num_tags + 1

    start_probability[index] = numerator / denominator


# ============================================================
# 9. END PROBABILITY
# ============================================================

end_probability = np.zeros(num_tags)


for tag in tags_list:

    index = tag_to_index[tag]

    total = sum(transition_counts[tag].values())

    numerator = transition_counts[tag][END] + 1

    denominator = total + num_tags + 1

    end_probability[index] = numerator / denominator


# ============================================================
# 10. BUILD EMISSION PROBABILITY FUNCTION
# ============================================================

def emission_probability(tag, word):

    word = word.lower()

    numerator = emission_counts[tag][word] + 1

    denominator = tag_counts[tag] + vocabulary_size

    return numerator / denominator


# ============================================================
# 11. CREATE LOG PROBABILITY MATRICES
# ============================================================

log_transition_matrix = np.log(transition_matrix)

log_start_probability = np.log(start_probability)

log_end_probability = np.log(end_probability)


# ============================================================
# 12. VECTORIZED VITERBI ALGORITHM
# ============================================================

def viterbi(words):

    if len(words) == 0:
        return []

    words = [word.lower() for word in words]

    number_of_words = len(words)

    # --------------------------------------------------------
    # Viterbi table
    # Rows    = POS tags
    # Columns = words
    # --------------------------------------------------------

    viterbi_table = np.full(
        (num_tags, number_of_words),
        -np.inf
    )

    # Backpointer
    backpointer = np.zeros(
        (num_tags, number_of_words),
        dtype=int
    )

    # --------------------------------------------------------
    # FIRST WORD
    # --------------------------------------------------------

    first_word = words[0]

    emission_values = np.array([
        math.log(emission_probability(tag, first_word))
        for tag in tags_list
    ])

    viterbi_table[:, 0] = (
        log_start_probability + emission_values
    )

    # --------------------------------------------------------
    # REMAINING WORDS
    # --------------------------------------------------------

    for position in range(1, number_of_words):

        word = words[position]

        # Emission probability for current word
        emission_values = np.array([
            math.log(emission_probability(tag, word))
            for tag in tags_list
        ])

        # ----------------------------------------------------
        # Vectorized calculation
        #
        # previous scores:
        # viterbi_table[:, position-1]
        #
        # transition:
        # log_transition_matrix
        # ----------------------------------------------------

        scores = (
            viterbi_table[:, position - 1][:, np.newaxis]
            + log_transition_matrix
        )

        # Best previous tag for every current tag
        best_previous_tags = np.argmax(
            scores,
            axis=0
        )

        # Best probability for every current tag
        best_scores = np.max(
            scores,
            axis=0
        )

        # Add emission probability
        viterbi_table[:, position] = (
            best_scores + emission_values
        )

        # Store backpointer
        backpointer[:, position] = best_previous_tags

    # ========================================================
    # FIND BEST FINAL TAG
    # ========================================================

    final_scores = (
        viterbi_table[:, -1]
        + log_end_probability
    )

    best_final_tag = np.argmax(final_scores)

    # ========================================================
    # BACKTRACKING
    # ========================================================

    best_path = [best_final_tag]

    for position in range(number_of_words - 1, 0, -1):

        previous_tag = backpointer[
            best_path[-1],
            position
        ]

        best_path.append(previous_tag)

    # Reverse the path
    best_path.reverse()

    # Convert indexes to POS tags
    predicted_tags = [
        index_to_tag[index]
        for index in best_path
    ]

    return predicted_tags


# ============================================================
# 13. TEST USER ENTERED SENTENCE
# ============================================================

sentence = input("\nEnter sentence: ")

words = sentence.split()

predicted_tags = viterbi(words)


print("\nPredicted POS Tags:")
print("-------------------")

for word, tag in zip(words, predicted_tags):

    print(f"{word} --> {tag}")


# ============================================================
# 14. EVALUATION ON TEST DATA
# ============================================================

correct = 0
total = 0


print("\nEvaluating on test dataset...")


for words, actual_tags in test_data:

    predicted_tags = viterbi(words)

    for predicted, actual in zip(
        predicted_tags,
        actual_tags
    ):

        if predicted == actual:
            correct += 1

        total += 1


# ============================================================
# 15. OVERALL ACCURACY
# ============================================================

accuracy = (correct / total) * 100


print("\n========================================")
print("          EVALUATION REPORT")
print("========================================")

print("Total test tokens:", total)

print("Correct Predictions:", correct)

print("Incorrect Predictions:", total - correct)

print(f"POS Tagging Accuracy: {accuracy:.2f}%")

print("========================================")


# ============================================================
# 16. PER-POS TAG ACCURACY
# ============================================================

tag_correct = Counter()
tag_total = Counter()


for words, actual_tags in test_data:

    predicted_tags = viterbi(words)

    for predicted, actual in zip(
        predicted_tags,
        actual_tags
    ):

        tag_total[actual] += 1

        if predicted == actual:
            tag_correct[actual] += 1


print("\nPer-POS Tag Accuracy:")
print("---------------------")


for tag in sorted(tag_total):

    tag_accuracy = (
        tag_correct[tag] /
        tag_total[tag]
    ) * 100

    print(
        f"{tag:8s}: "
        f"{tag_accuracy:6.2f}% "
        f"({tag_correct[tag]}/{tag_total[tag]})"
    )


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n========================================")
print("              SUMMARY")
print("========================================")

print("Training sentences :", len(train_data))
print("Testing sentences  :", len(test_data))
print("Number of POS tags  :", num_tags)
print("Vocabulary size     :", vocabulary_size)
print(f"Final Accuracy      : {accuracy:.2f}%")

print("========================================")

Downloaded: en_ewt-ud-train.conllu
Downloaded: en_ewt-ud-test.conllu

Training sentences: 12544
Testing sentences: 2077

POS Tags:
['PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'DET', 'ADP', 'AUX', 'PRON', 'PART', 'SCONJ', 'NUM', 'ADV', 'CCONJ', 'INTJ', 'X', 'SYM']

Vocabulary size: 16655



Enter sentence:  The student reads a book



Predicted POS Tags:
-------------------
The --> DET
student --> NOUN
reads --> ADP
a --> DET
book --> NOUN

Evaluating on test dataset...

          EVALUATION REPORT
Total test tokens: 25094
Correct Predictions: 21457
Incorrect Predictions: 3637
POS Tagging Accuracy: 85.51%
